In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [llama_stack_client]lama_stack_client]


In [ ]:
import os
import sys
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
import logging
import requests
from io import BytesIO

In [11]:
sys.path.append('..')
# Load environment variables from .env file
load_dotenv()

logger = logging.getLogger(__name__)
logger.setLevel("INFO")

# Initialize the Llama Stack client
client = LlamaStackClient(
    base_url=os.getenv("LLAMA_STACK_SERVER_URL", "http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321")
)

file_path = "data/Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv"
url = "https://www.openshift.guide/openshift-guide-screen.pdf"
vector_db_skus_name = "skus_rh_vector_db"
vector_db_ocp_name = "ocp_rh_vector_db"

logger.info("Connected to Llama Stack server")

INFO:__main__:Connected to Llama Stack server


In [15]:
# Upload a document
file = client.files.create(
    file=open(file_path, "rb"),
    purpose="assistants",
)
print(f"Uploaded: {file.id}")

# Create a vector store and index the file
vector_store_skus = client.vector_stores.create(
    name=vector_db_skus_name,
    extra_body={
        "provider_id": "milvus",
        "embedding_model": "sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
        "embedding_dimension": 768,
    },
)

client.vector_stores.files.create(vector_store_id=vector_store_skus.id, file_id=file.id)

print(f"Vector store SKUs: {vector_store_skus.id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Uploaded: file-64ddb8f3360d42569162fd68bffb1d1f


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_36cc39f1-6b55-4e75-8663-9f2c1607dee0/files "HTTP/1.1 200 OK"


Vector store SKUs: vs_36cc39f1-6b55-4e75-8663-9f2c1607dee0


In [ ]:
response = requests.get(url)
file_buffer = BytesIO(response.content)
file_buffer.name = "openshift-guide-screen.pdf"

# Upload a document
file_url = client.files.create(
    file=file_buffer,
    purpose="assistants",
)
print(f"Uploaded: {file_url.id}")

# Create a vector store and index the file
vector_store_ocp = client.vector_stores.create(
    name=vector_db_ocp_name,
    extra_body={
        "provider_id": "milvus",
        "embedding_model": "sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
        "embedding_dimension": 768,
    },
    file_ids=[file_url.id],
)
print(f"Vector store OCP: {vector_store_ocp.id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"


Uploaded: file-3cce75de008a4206aad9ae5314d54998


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector store Web: vs_467b2eb2-9fb7-4e0b-9ad0-0d3fa2903013


In [ ]:
query = "List of Red Hat OpenShift SKUs"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_skus.id],
    }],
)

logger.info(f"RAG Query from {vector_db_skus_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from skus_rh_vector_db - Result: 
The Red Hat OpenShift SKUs are not explicitly listed in the knowledge search results. However, based on general knowledge, the Red Hat OpenShift SKUs include:

* OpenShift Container Platform
* OpenShift Dedicated
* OpenShift Online

Please note that the list of SKUs may not be exhaustive and is subject to change based on Red Hat's product offerings.


In [13]:
query = "What is Red Hat OpenShift?"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_ocp.id],
    }],
)

logger.info(f"RAG Query from {vector_db_ocp_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from ocp_rh_vector_db - Result: 
Red Hat OpenShift is a container application platform that allows developers to easily build, deploy, and manage applications in a containerized environment. It provides a flexible and scalable way to deploy and manage applications, and supports a wide range of programming languages and frameworks. OpenShift also provides a range of tools and services to help developers manage their applications, including automated deployment, scaling, and monitoring.
